# 03 — Content-Based

**Group 9 contributors**

- ANDREA BASTIEN SAXOD
- CLOÉ KARINA CHAPOTOT
- CONSTANTIN NICOLA HATECKE
- MARCELA MENDES GIMENES FUNABASHI
- MATIAS VICENTE AREVALO MARTINEZ
- VITTORIO FIALDINI

## Objective

Recommend players based on what they are, not how they have been used. The feature vector captures role and performance; cosine similarity to a team profile produces the ranking.

## Inputs

- `players_scored` from notebook 01 for the real-only feature space.
- `players_master_synthetic_augmented.csv` for the synthetic-augmented feature space.

## Method summary

1. Build two feature spaces. The real-only vector uses z-scored FBref per-90 rates plus role dummies and age terms. The synthetic-augmented vector adds explainable trait dimensions such as `syn_trait_finishing` and `syn_xg_p90`.
2. L2-normalise every player vector so cosine similarity is a dot product.
3. Construct a team profile as the minutes-weighted centroid of the squad's players at the target position.
4. Score all candidates from the previous season by cosine similarity, excluding players already at the club.

## Conceptual note on "replicate" versus "complement"

A squad-centroid profile answers "find players similar to who this club already has." For scouting that is not always what a club actually wants. A club with a good squad of a given profile is often hunting for the missing piece, not another copy of the same piece. We address this by adding a second mode, `complement`, that scores a gap vector defined as

    direction = (1 - alpha) * squad_profile + alpha * (peer_profile - squad_profile)

where `peer_profile` is the minutes-weighted centroid of the top-quintile players at the same position in the same league. `alpha` controls how much we tilt away from "more of the same" and towards "what peers have that we lack." This is one pragmatic fix for the concern that a pure replicate-the-squad profile biases the model towards duplication. It is not a full transfer-history model, which would need an arrivals dataset the real-only track does not have.

## Key assumptions

- The real-only feature space covers enough role-defining statistics to separate creators from ball-winners.
- A minutes-weighted centroid is a reasonable summary of a squad at a position.
- The synthetic-augmented feature space is used only as a proof-of-concept enrichment layer.

## Outputs

- Role-subtype coverage of the feature space.
- Similar-players sanity check anchored on Rodri in 2022-23.
- Arsenal MF shortlist in the real-only space.
- Arsenal MF and Liverpool DF shortlists in the synthetic-augmented space.
- Replicate-versus-complement comparison on the Arsenal MF shortlist.


In [1]:
import warnings, json
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def locate_project_root():
    """Find a root directory that contains the expected data files.
    Supports Colab layout (/content/Recommendation_engine_group), a data/ folder
    with real_uploaded and synthetic subfolders, or a flat upload folder.
    """
    candidates = [
        Path('/content/Recommendation_engine_group'),
        Path('/content'),
        Path('/mnt/user-data/uploads'),
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path('/home/claude'),
    ]
    for base in candidates:
        if not base.exists():
            continue
        has_structured = (base / 'data').exists() and (
            (base / 'data' / 'real').exists()
            or (base / 'data' / 'real_uploaded').exists()
            or (base / 'data' / 'synthetic').exists()
        )
        has_flat = (base / 'players_master_with_market_values.csv').exists() or (
            base / 'players_master_synthetic_augmented.csv'
        ).exists()
        if has_structured or has_flat:
            return base
    raise FileNotFoundError('Could not locate project root with the expected data files.')

ROOT = locate_project_root()
print('Using project root:', ROOT)

def read_first_existing(paths, **kwargs):
    last_err = None
    for p in paths:
        p = Path(p)
        if p.exists():
            try:
                return pd.read_csv(p, **kwargs)
            except Exception as e:
                last_err = e
    if last_err is not None:
        raise last_err
    raise FileNotFoundError(f'None of the candidate paths exist: {paths}')

def load_real_tables(root=ROOT):
    players = read_first_existing([
        root / 'data' / 'real_uploaded' / 'players_master_with_market_values.csv',
        root / 'data' / 'real' / 'players_master_with_market_values.csv.gz',
        root / 'players_master_with_market_values.csv',
    ], low_memory=False)
    teams = read_first_existing([
        root / 'data' / 'real_uploaded' / 'teams_master_with_market_values.csv',
        root / 'data' / 'real' / 'teams_master_with_market_values.csv.gz',
        root / 'teams_master_with_market_values.csv',
    ], low_memory=False)
    return players, teams

def load_synthetic_tables(root=ROOT):
    players_aug = read_first_existing([
        root / 'data' / 'synthetic' / 'players_master_synthetic_augmented.csv',
        root / 'players_master_synthetic_augmented.csv',
    ], low_memory=False)
    teams_aug = read_first_existing([
        root / 'data' / 'synthetic' / 'teams_master_synthetic_augmented.csv',
        root / 'teams_master_synthetic_augmented.csv',
    ], low_memory=False)
    return players_aug, teams_aug

TRAIN_SEASONS = ['2018-19','2019-20','2020-21','2021-22']
VAL_SEASON = '2022-23'
TEST_SEASON = '2023-24'
PREV_SEASON = {'2019-20':'2018-19','2020-21':'2019-20','2021-22':'2020-21','2022-23':'2021-22','2023-24':'2022-23'}

PER90_MAP = {
    'misc__Performance__Int':'interceptions_p90',
    'misc__Performance__TklW':'tackles_won_p90',
    'misc__Performance__Fls':'fouls_committed_p90',
    'misc__Performance__Fld':'fouls_won_p90',
    'misc__Performance__Crs':'crosses_p90',
    'misc__Performance__Off':'offsides_p90',
}

REAL_SCORE_FEATURES = [
    'standard__Per 90 Minutes__G-PK',
    'standard__Per 90 Minutes__Ast',
    'shooting__Standard__Sh/90',
    'shooting__Standard__SoT/90',
    'shooting__Standard__G/Sh',
    'shooting__Standard__SoT%',
    'interceptions_p90',
    'tackles_won_p90',
    'crosses_p90',
    'fouls_won_p90',
    'offsides_p90',
    'playing_time__Team Success__+/-90',
    'playing_time__Team Success__On-Off',
    'playing_time__Playing Time__Min%',
]

CONTENT_REAL_VECTOR = [
    'standard__Per 90 Minutes__G-PK_z',
    'standard__Per 90 Minutes__Ast_z',
    'shooting__Standard__Sh/90_z',
    'shooting__Standard__SoT/90_z',
    'shooting__Standard__G/Sh_z',
    'shooting__Standard__SoT%_z',
    'interceptions_p90_z',
    'tackles_won_p90_z',
    'crosses_p90_z',
    'fouls_won_p90_z',
    'playing_time__Team Success__+/-90_z',
    'playing_time__Team Success__On-Off_z',
    'playing_time__Playing Time__Min%_z',
]

SYNTH_VECTOR_COLS = [
    'syn_trait_finishing','syn_trait_creativity','syn_trait_progression','syn_trait_ball_carrying',
    'syn_trait_defensive_intensity','syn_trait_press_resistance','syn_trait_aerial_physicality',
    'syn_trait_offball_threat','syn_xg_p90','syn_xa_p90','syn_xgi_p90','syn_progressive_passes_p90',
    'syn_progressive_carries_p90','syn_key_passes_p90','syn_tackles_won_p90','syn_interceptions_p90',
    'syn_pressures_applied_p90','syn_ball_retention_pct','syn_availability_pct'
]

def _primary_position(pos):
    if pd.isna(pos):
        return 'UNK'
    return str(pos).split(',')[0].strip()

def add_position_columns(players):
    out = players.copy()
    out['pos_primary'] = out['pos'].apply(_primary_position)
    out['pos_family'] = out['pos_primary'].map({'GK':'GK','DF':'DF','MF':'MF','FW':'FW'}).fillna('UNK')
    return out

def add_per90_columns(players):
    out = add_position_columns(players)
    for raw_col, new_col in PER90_MAP.items():
        if raw_col in out.columns:
            out[new_col] = out[raw_col] / out['nineties'].clip(lower=0.1)
        else:
            out[new_col] = np.nan
    return out

def filtered_players(players):
    out = add_per90_columns(players)
    outfield = out['pos_family'].isin(['DF','MF','FW']) & (out['minutes_played'] >= 600)
    keepers = (out['pos_family'] == 'GK') & (out['minutes_played'] >= 900)
    return out.loc[outfield | keepers].copy().reset_index(drop=True)

def safe_group_zscore(df, col, group_cols):
    grouped = df.groupby(group_cols)[col]
    mean = grouped.transform('mean')
    std = grouped.transform('std').replace(0, np.nan)
    z = (df[col] - mean) / std
    return z.replace([np.inf,-np.inf], np.nan).fillna(0.0)

def infer_role_subtype(players_f):
    out = players_f.copy()
    out['creator_proxy'] = out['standard__Per 90 Minutes__Ast_z'] + 0.7*out['crosses_p90_z'] + 0.4*out['fouls_won_p90_z']
    out['defender_proxy'] = out['interceptions_p90_z'] + 0.9*out['tackles_won_p90_z'] - 0.2*out['crosses_p90_z']
    out['striker_proxy'] = out['shooting__Standard__Sh/90_z'] + 0.9*out['standard__Per 90 Minutes__G-PK_z'] + 0.5*out['offsides_p90_z']
    out['wing_proxy'] = out['standard__Per 90 Minutes__Ast_z'] + 0.8*out['crosses_p90_z'] + 0.4*out['fouls_won_p90_z']
    role = []
    for _, row in out.iterrows():
        fam = row['pos_family']
        if fam == 'GK':
            role.append('Goalkeeper')
        elif fam == 'DF':
            role.append('Full-back / Wing-back' if (row['crosses_p90_z'] + row['standard__Per 90 Minutes__Ast_z']) > 0.3 else 'Centre-back')
        elif fam == 'MF':
            if row['creator_proxy'] - row['defender_proxy'] > 0.45:
                role.append('Creator / Advanced MF')
            elif row['defender_proxy'] - row['creator_proxy'] > 0.45:
                role.append('Ball-winning / Holding MF')
            else:
                role.append('Box-to-box / Hybrid MF')
        elif fam == 'FW':
            if row['striker_proxy'] - row['wing_proxy'] > 0.5:
                role.append('Striker')
            elif row['wing_proxy'] - row['striker_proxy'] > 0.2:
                role.append('Winger / Support Forward')
            else:
                role.append('Second Striker / Hybrid Forward')
        else:
            role.append('Other')
    out['role_subtype'] = role
    return out

def build_real_scoring_table(players, weights_override=None):
    """Build the scored player-season table.
    weights_override lets the sensitivity analysis perturb the composite without
    duplicating the whole function. Keys expected: 'FW', 'MF', 'DF' each mapped to
    a dict of {feature_z_col: weight}.
    """
    out = filtered_players(players)
    for col in REAL_SCORE_FEATURES:
        if col not in out.columns:
            out[col] = np.nan
        out[col] = out.groupby(['league','season_label','pos_family'])[col].transform(lambda s: s.fillna(s.median()))
        out[col] = out[col].fillna(out[col].median())
        out[f'{col}_z'] = safe_group_zscore(out, col, ['season_label','league','pos_family'])
    out = infer_role_subtype(out)

    mv = out['tm_market_value_eur_resolved'].fillna(out['tm_market_value_eur_resolved'].median())
    out['log_market_value'] = np.log1p(mv)
    out['log_market_value_z'] = safe_group_zscore(out, 'log_market_value', ['season_label','pos_family'])
    out['age_curve'] = np.exp(-((out['age'] - 26.0) ** 2) / (2 * 5.0**2))

    default_weights = {
        'FW': {
            'standard__Per 90 Minutes__G-PK_z': 0.36,
            'standard__Per 90 Minutes__Ast_z':  0.18,
            'shooting__Standard__SoT/90_z':     0.18,
            'shooting__Standard__G/Sh_z':       0.10,
            'fouls_won_p90_z':                  0.08,
            'playing_time__Team Success__+/-90_z': 0.10,
        },
        'MF': {
            'standard__Per 90 Minutes__Ast_z':  0.24,
            'standard__Per 90 Minutes__G-PK_z': 0.16,
            'interceptions_p90_z':              0.20,
            'tackles_won_p90_z':                0.16,
            'fouls_won_p90_z':                  0.10,
            'crosses_p90_z':                    0.06,
            'playing_time__Team Success__+/-90_z': 0.08,
        },
        'DF': {
            'interceptions_p90_z':              0.32,
            'tackles_won_p90_z':                0.28,
            'crosses_p90_z':                    0.10,
            'playing_time__Team Success__+/-90_z': 0.15,
            'playing_time__Team Success__On-Off_z': 0.15,
        },
    }
    weights = weights_override if weights_override is not None else default_weights

    out['real_quality_score'] = 0.0
    for fam, ws in weights.items():
        mask = out['pos_family'] == fam
        if not mask.any():
            continue
        score = np.zeros(mask.sum())
        for col, w in ws.items():
            if col in out.columns:
                score = score + w * out.loc[mask, col].to_numpy()
        out.loc[mask, 'real_quality_score'] = score

    gk = out['pos_family'] == 'GK'
    if gk.any():
        for col in ['keeper__Performance__Save%','keeper__Performance__CS%','keeper__Performance__GA90']:
            if col in out.columns:
                out[col] = out.groupby(['league','season_label'])[col].transform(lambda s: s.fillna(s.median()))
                out[col] = out[col].fillna(out[col].median())
                out[f'{col}_z'] = safe_group_zscore(out, col, ['season_label','league'])
            else:
                out[f'{col}_z'] = 0.0
        out.loc[gk, 'real_quality_score'] = (
            0.40*out.loc[gk,'keeper__Performance__Save%_z'] +
            0.30*out.loc[gk,'keeper__Performance__CS%_z'] -
            0.30*out.loc[gk,'keeper__Performance__GA90_z']
        )

    out['value_adjusted_score'] = out['real_quality_score'] - 0.22*out['log_market_value_z']
    out['trajectory_adjusted_score'] = out['real_quality_score'] * (0.6 + 0.4*out['age_curve'])
    return out

def top_nonpersonalized(players_scored, season=TEST_SEASON, pos_family='FW', score_col='real_quality_score', top_n=10):
    cols = ['player','team','age','pos_family','role_subtype','tm_market_value_eur_resolved',score_col]
    return (
        players_scored[(players_scored['season_label']==season) & (players_scored['pos_family']==pos_family)]
        .sort_values(score_col, ascending=False)[cols]
        .rename(columns={'tm_market_value_eur_resolved':'market_value_eur'})
        .head(top_n)
        .reset_index(drop=True)
    )

def read_synthetic_players(root=ROOT):
    return read_first_existing([
        root / 'data' / 'synthetic' / 'players_master_synthetic_augmented.csv',
        root / 'players_master_synthetic_augmented.csv',
    ], low_memory=False)

def l2_normalise(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return matrix / norms

def build_content_features(players_scored, players_aug=None, use_synthetic=False):
    df = players_scored.copy()
    feature_cols = list(CONTENT_REAL_VECTOR)
    if use_synthetic and players_aug is not None:
        keep = ['player_key','team','season_label'] + [c for c in SYNTH_VECTOR_COLS if c in players_aug.columns]
        df = df.merge(players_aug[keep], on=['player_key','team','season_label'], how='left')
        for col in [c for c in SYNTH_VECTOR_COLS if c in df.columns]:
            df[col] = df.groupby(['season_label','pos_family'])[col].transform(lambda s: s.fillna(s.median()))
            df[col] = df[col].fillna(df[col].median())
            df[f'{col}_z'] = safe_group_zscore(df, col, ['season_label','pos_family'])
            feature_cols.append(f'{col}_z')
    role_dummies = pd.get_dummies(df['role_subtype'], prefix='role').astype(float)
    df['age_centered'] = (df['age'] - 26.0) / 5.0
    df['age_sq'] = df['age_centered'] ** 2
    out = pd.concat([
        df[['player_key','team','league','season_label','player','age','minutes_played','pos_family','role_subtype']],
        df[[c for c in feature_cols if c in df.columns]].fillna(0.0),
        role_dummies,
        df[['age_centered','age_sq']]
    ], axis=1)
    out = out.loc[:, ~out.columns.duplicated()]
    final_cols = [c for c in feature_cols if c in out.columns] + list(role_dummies.columns) + ['age_centered','age_sq']
    return out, final_cols

def team_profile(feature_df, feature_cols, team, season, pos_family):
    squad = feature_df[(feature_df['team']==team) & (feature_df['season_label']==season) & (feature_df['pos_family']==pos_family)]
    if squad.empty:
        return None
    weights = squad['minutes_played'].to_numpy(dtype=float)
    if weights.sum() <= 0:
        weights = np.ones(len(squad), dtype=float)
    weights = weights / weights.sum()
    matrix = squad[feature_cols].to_numpy(dtype=float)
    vec = matrix.T @ weights
    norm = np.linalg.norm(vec)
    return vec if norm == 0 else vec / norm

def peer_profile(feature_df, feature_cols, team, season, pos_family, percentile=0.8):
    '''Build a profile of the top-tier teams' players at this position.
    Used as the "aspirational" direction for the gap-vector variant.'''
    same_league = feature_df[(feature_df['season_label']==season) & (feature_df['pos_family']==pos_family)].copy()
    my_league = feature_df.loc[(feature_df['team']==team) & (feature_df['season_label']==season), 'league']
    if not my_league.empty:
        same_league = same_league[same_league['league']==my_league.iloc[0]]
    if same_league.empty:
        return None
    cutoff = same_league['minutes_played'].quantile(percentile)
    top = same_league[same_league['minutes_played'] >= cutoff]
    weights = top['minutes_played'].to_numpy(dtype=float)
    if weights.sum() <= 0:
        return None
    weights = weights / weights.sum()
    matrix = top[feature_cols].to_numpy(dtype=float)
    vec = matrix.T @ weights
    norm = np.linalg.norm(vec)
    return vec if norm == 0 else vec / norm

def similar_players(feature_df, feature_cols, player_name, season='2022-23', top_n=10):
    subset = feature_df[feature_df['season_label']==season].copy()
    anchor = subset[subset['player'].str.lower()==player_name.lower()].sort_values('minutes_played', ascending=False).head(1)
    if anchor.empty:
        return pd.DataFrame()
    anchor_row = anchor.iloc[0]
    candidates = subset[(subset['pos_family']==anchor_row['pos_family']) & (subset['player_key']!=anchor_row['player_key'])].copy()
    matrix = l2_normalise(candidates[feature_cols].to_numpy(dtype=float))
    anchor_vec = anchor[feature_cols].to_numpy(dtype=float)
    anchor_vec = anchor_vec / max(np.linalg.norm(anchor_vec), 1e-12)
    candidates['similarity'] = (matrix @ anchor_vec.reshape(-1,1)).ravel()
    return candidates.sort_values('similarity', ascending=False)[['player','team','age','pos_family','role_subtype','similarity']].head(top_n).reset_index(drop=True)

def content_recommendations(feature_df, feature_cols, players_scored, team,
                            target_season=TEST_SEASON, pos_family='FW', top_n=10,
                            mode='replicate', gap_weight=0.5):
    '''Two modes:
       - replicate: standard similarity to the squad profile (default, backward compatible).
       - complement: score a gap vector = peer_profile - current_squad_profile,
         which biases towards players who address the gap between what the club
         has and what top-tier clubs at the same position look like.
    '''
    prev = PREV_SEASON[target_season]
    profile = team_profile(feature_df, feature_cols, team, prev, pos_family)
    if profile is None:
        return pd.DataFrame()

    if mode == 'complement':
        peer = peer_profile(feature_df, feature_cols, team, prev, pos_family)
        if peer is None:
            direction = profile
        else:
            gap = peer - profile
            norm = np.linalg.norm(gap)
            gap = gap if norm == 0 else gap / norm
            direction = (1 - gap_weight) * profile + gap_weight * gap
            n = np.linalg.norm(direction)
            direction = direction if n == 0 else direction / n
    else:
        direction = profile

    candidates = (feature_df[feature_df['season_label']==prev]
                  .sort_values(['player_key','minutes_played'], ascending=[True, False])
                  .drop_duplicates('player_key')
                  .copy())
    candidates = candidates[candidates['pos_family']==pos_family].copy()
    prev_team_players = set(players_scored.loc[(players_scored['season_label']==prev) & (players_scored['team']==team), 'player_key'])
    candidates = candidates[~candidates['player_key'].isin(prev_team_players)].copy()
    matrix = l2_normalise(candidates[feature_cols].fillna(0.0).to_numpy(dtype=float))
    candidates['cb_score'] = (matrix @ direction.reshape(-1,1)).ravel()
    return candidates.sort_values('cb_score', ascending=False)[['player','team','age','pos_family','role_subtype','cb_score']].head(top_n).reset_index(drop=True)

players, teams = load_real_tables(ROOT)
players_aug = read_synthetic_players(ROOT)
players_scored = build_real_scoring_table(players)
real_features, real_feature_cols = build_content_features(players_scored)
synth_features, synth_feature_cols = build_content_features(players_scored, players_aug, use_synthetic=True)

summary = pd.DataFrame({
    'feature_space': ['real-only', 'synthetic-augmented'],
    'n_vectors': [len(real_features), len(synth_features)],
    'n_dimensions': [len(real_feature_cols), len(synth_feature_cols)],
})
display(summary)


Using project root: /mnt/user-data/uploads


,feature_space,n_vectors,n_dimensions
0,real-only,11093,24
1,synthetic-augmented,11093,43


## Evidence: feature columns and normalisation sanity check

The feature set is explicit here. The L2-normalisation check confirms that after building the vectors, every row has unit norm so cosine similarity equals the dot product.


In [2]:
real_feature_listing = pd.DataFrame({'real_feature_col': real_feature_cols})
synth_feature_listing = pd.DataFrame({'synth_feature_col': synth_feature_cols})
display(real_feature_listing.head(20))
print(f'Total real features: {len(real_feature_cols)}')
print(f'Total synthetic features: {len(synth_feature_cols)}')

# Norm check on a sample of 500 rows.
sample = real_features.sample(500, random_state=0)
matrix = sample[real_feature_cols].to_numpy(dtype=float)
raw_norms = np.linalg.norm(matrix, axis=1)
normalised = l2_normalise(matrix)
post_norms = np.linalg.norm(normalised, axis=1)
print(f'Raw vector norms: min {raw_norms.min():.3f}, median {np.median(raw_norms):.3f}, max {raw_norms.max():.3f}')
print(f'After L2 normalisation: min {post_norms.min():.3f}, median {np.median(post_norms):.3f}, max {post_norms.max():.3f}')


,real_feature_col
0,standard__Per 90 Minutes__G-PK_z
1,standard__Per 90 Minutes__Ast_z
2,shooting__Standard__Sh/90_z
3,shooting__Standard__SoT/90_z
4,shooting__Standard__G/Sh_z
5,shooting__Standard__SoT%_z
6,interceptions_p90_z
7,tackles_won_p90_z
8,crosses_p90_z
9,fouls_won_p90_z


Total real features: 24
Total synthetic features: 43
Raw vector norms: min 1.746, median 3.665, max 9.510
After L2 normalisation: min 1.000, median 1.000, max 1.000


## Role subtype distribution

In [3]:
display(players_scored['role_subtype'].value_counts().rename_axis('role_subtype').reset_index(name='rows'))

,role_subtype,rows
0,Centre-back,2597
1,Ball-winning / Holding MF,2091
2,Creator / Advanced MF,2009
3,Full-back / Wing-back,1216
4,Winger / Support Forward,816
5,Striker,777
6,Goalkeeper,718
7,Box-to-box / Hybrid MF,601
8,Second Striker / Hybrid Forward,268


## Similar players to Rodri (real-only vectors, 2022-23)

Sanity check. If the vectors encode role correctly the nearest neighbours should be other holding or deep-lying midfielders, not wingers.


In [4]:
display(similar_players(real_features, real_feature_cols, player_name='Rodri',
                        season='2022-23', top_n=10))

,player,team,age,pos_family,role_subtype,similarity
0,Manuel Locatelli,Juventus,24,MF,Ball-winning / Holding MF,0.746782
1,Andre-Frank Zambo Anguissa,Napoli,26,MF,Ball-winning / Holding MF,0.735976
2,Joe Willock,Newcastle United,22,MF,Box-to-box / Hybrid MF,0.730882
3,Thomas Partey,Arsenal,29,MF,Ball-winning / Holding MF,0.729373
4,Fabinho,Liverpool,28,MF,Ball-winning / Holding MF,0.723669
5,Koke,Atlético Madrid,30,MF,Ball-winning / Holding MF,0.688509
6,Rade Krunić,Milan,28,MF,Ball-winning / Holding MF,0.679355
7,Youssouf Fofana,Monaco,23,MF,Ball-winning / Holding MF,0.678628
8,Stanislav Lobotka,Napoli,27,MF,Ball-winning / Holding MF,0.675220
9,Nordi Mukiele,Paris Saint-Germain,24,MF,Ball-winning / Holding MF,0.671805


## Arsenal midfield targets from the real-only content space

Replicate mode. The squad profile is the minutes-weighted centroid of Arsenal's 2022-23 midfielders.


In [5]:
display(content_recommendations(real_features, real_feature_cols, players_scored,
                                team='Arsenal', target_season=TEST_SEASON,
                                pos_family='MF', top_n=10, mode='replicate'))

,player,team,age,pos_family,role_subtype,cb_score
0,İlkay Gündoğan,Manchester City,31,MF,Creator / Advanced MF,0.858699
1,Elif Elmas,Napoli,22,MF,Creator / Advanced MF,0.777743
2,Federico Valverde,Real Madrid,24,MF,Creator / Advanced MF,0.734015
3,Bernardo Silva,Manchester City,27,MF,Creator / Advanced MF,0.699125
4,Seko Fofana,Lens,27,MF,Creator / Advanced MF,0.681806
5,Thomas Müller,Bayern Munich,32,MF,Creator / Advanced MF,0.670990
6,Yannick Gerhardt,Wolfsburg,28,MF,Creator / Advanced MF,0.667774
7,Florian Sotoca,Lens,31,MF,Creator / Advanced MF,0.657108
8,Kaoru Mitoma,Brighton,25,MF,Creator / Advanced MF,0.656271
9,Rodri,Manchester City,26,MF,Ball-winning / Holding MF,0.653526


## Arsenal midfield targets from the synthetic-augmented content space

In [6]:
display(content_recommendations(synth_features, synth_feature_cols, players_scored,
                                team='Arsenal', target_season=TEST_SEASON,
                                pos_family='MF', top_n=10, mode='replicate'))

,player,team,age,pos_family,role_subtype,cb_score
0,İlkay Gündoğan,Manchester City,31,MF,Creator / Advanced MF,0.860111
1,Seko Fofana,Lens,27,MF,Creator / Advanced MF,0.769511
2,Luis Alberto,Lazio,29,MF,Creator / Advanced MF,0.748588
3,Thomas Müller,Bayern Munich,32,MF,Creator / Advanced MF,0.736893
4,Sergej Milinković-Savić,Lazio,27,MF,Creator / Advanced MF,0.732146
5,Pascal Groß,Brighton,31,MF,Creator / Advanced MF,0.731679
6,Pedri,Barcelona,19,MF,Box-to-box / Hybrid MF,0.730804
7,Florian Sotoca,Lens,31,MF,Creator / Advanced MF,0.729508
8,Nicolò Barella,Inter,25,MF,Creator / Advanced MF,0.722227
9,Jonathan Ikone,Fiorentina,24,MF,Creator / Advanced MF,0.715610


## Liverpool defensive targets from the synthetic-augmented content space

In [7]:
display(content_recommendations(synth_features, synth_feature_cols, players_scored,
                                team='Liverpool', target_season=TEST_SEASON,
                                pos_family='DF', top_n=10, mode='replicate'))

,player,team,age,pos_family,role_subtype,cb_score
0,Hamari Traoré,Rennes,30,DF,Full-back / Wing-back,0.890009
1,Raphaël Guerreiro,Dortmund,28,DF,Full-back / Wing-back,0.866074
2,Jordi Alba,Barcelona,33,DF,Full-back / Wing-back,0.831657
3,Giovanni Di Lorenzo,Napoli,28,DF,Full-back / Wing-back,0.821107
4,Pervis Estupiñán,Brighton,24,DF,Full-back / Wing-back,0.810417
5,Alfonso Pedraza,Villarreal,26,DF,Full-back / Wing-back,0.779422
6,José Luis Gayà,Valencia,27,DF,Full-back / Wing-back,0.755478
7,Tyronne Ebuehi,Empoli,26,DF,Full-back / Wing-back,0.754071
8,Alphonso Davies,Bayern Munich,21,DF,Full-back / Wing-back,0.740692
9,Cristiano Biraghi,Fiorentina,29,DF,Full-back / Wing-back,0.740309


## Replicate versus complement: Arsenal midfielders

This compares the default "find players similar to the current squad" view with a gap-vector variant that tilts the target towards the delta between the squad and the top-quintile peer group. Names that only appear in the complement list are candidates the replicate profile was missing because it was duplicating strengths the squad already has. Names in both lists are robust recommendations regardless of framing.

`alpha = 0.5` is the tilt weight. Lower values stay close to the squad profile; higher values push harder towards the peer gap.


In [8]:
rep = content_recommendations(real_features, real_feature_cols, players_scored,
                              team='Arsenal', target_season=TEST_SEASON,
                              pos_family='MF', top_n=10, mode='replicate')
com = content_recommendations(real_features, real_feature_cols, players_scored,
                              team='Arsenal', target_season=TEST_SEASON,
                              pos_family='MF', top_n=10, mode='complement', gap_weight=0.5)

rep_set = set(rep['player'])
com_set = set(com['player'])
comparison = pd.DataFrame({
    'bucket': ['in both', 'only in replicate', 'only in complement'],
    'count': [len(rep_set & com_set), len(rep_set - com_set), len(com_set - rep_set)],
    'names': [
        ', '.join(sorted(rep_set & com_set)),
        ', '.join(sorted(rep_set - com_set)),
        ', '.join(sorted(com_set - rep_set)),
    ],
})
display(comparison)
print('\nReplicate shortlist:')
display(rep)
print('Complement shortlist:')
display(com)


,bucket,count,names
0,in both,1,Florian Sotoca
1,only in replicate,9,"Bernardo Silva, Elif Elmas, Federico Valverde,..."
2,only in complement,9,"Alex Iwobi, Bruno Fernandes, Carlos Augusto, D..."



Replicate shortlist:


,player,team,age,pos_family,role_subtype,cb_score
0,İlkay Gündoğan,Manchester City,31,MF,Creator / Advanced MF,0.858699
1,Elif Elmas,Napoli,22,MF,Creator / Advanced MF,0.777743
2,Federico Valverde,Real Madrid,24,MF,Creator / Advanced MF,0.734015
3,Bernardo Silva,Manchester City,27,MF,Creator / Advanced MF,0.699125
4,Seko Fofana,Lens,27,MF,Creator / Advanced MF,0.681806
5,Thomas Müller,Bayern Munich,32,MF,Creator / Advanced MF,0.670990
6,Yannick Gerhardt,Wolfsburg,28,MF,Creator / Advanced MF,0.667774
7,Florian Sotoca,Lens,31,MF,Creator / Advanced MF,0.657108
8,Kaoru Mitoma,Brighton,25,MF,Creator / Advanced MF,0.656271
9,Rodri,Manchester City,26,MF,Ball-winning / Holding MF,0.653526


Complement shortlist:


,player,team,age,pos_family,role_subtype,cb_score
0,Alex Iwobi,Everton,26,MF,Creator / Advanced MF,0.758564
1,Jarrod Bowen,West Ham United,25,MF,Creator / Advanced MF,0.733563
2,Bruno Fernandes,Manchester Utd,27,MF,Creator / Advanced MF,0.723563
3,Teun Koopmeiners,Atalanta,24,MF,Creator / Advanced MF,0.715756
4,James Ward-Prowse,Southampton,27,MF,Creator / Advanced MF,0.711538
5,Florian Sotoca,Lens,31,MF,Creator / Advanced MF,0.706319
6,Douglas Luiz,Aston Villa,24,MF,Box-to-box / Hybrid MF,0.701008
7,Álvaro García,Rayo Vallecano,29,MF,Creator / Advanced MF,0.699444
8,Carlos Augusto,Monza,23,MF,Box-to-box / Hybrid MF,0.698937
9,Sergi Darder,Espanyol,28,MF,Creator / Advanced MF,0.693316


## Interpretation

The similar-players check on Rodri returns other holding midfielders, which is what we want from a role-aware vector space. The squad-centroid recommendations tend to look like "more of what the club already has," which the complement variant partially corrects by tilting the target vector towards the gap between the club and its peer group. Which view a scout prefers depends on their intent: reinforce a strength, or close a gap.

The synthetic descriptors are a proof-of-concept enrichment layer. The honest benchmark remains the real-only track in `05_evaluation.ipynb`.
